# RG3 F1目標: 학습 구간 내부 앙상블·임계값 공동 선택

고정 RG3 RH 정상601/가스19. 과거 std4 원래 RF 설정(200/depth8/leaf3/balanced_subsample/42)을 같은7개 관측구간에서 재평가한다. 해당 단독RF는 비교용이며 최종 후보는 std4 깊은 RF/ET 앙상블, 조건부 정상순위75 앙상블, 절대상태10 앙상블(각4모델)의 가중 합이다.

가중치 .25단위 simplex15개를 사전 고정, 내부 OOF 예측에서 F1 최적 임계값을 선택. 목적 우선순위는 현재 사용자 지시 F1 목표에 맞춰 F1, 동률 탐지율, 동률 오탐 적은 순. 기존 회귀/순위 실험의 탐지율 우선 선택과 구분. 평가 라벨/구간은 정상 이웃·스케일·분류·가중치·임계값 학습 모두 제외. ref 기준 모델 캐시에는 전체 센서 행을 변환/예측하나 fit/정상 이웃은 ref만 사용하고 평가 라벨은 출력 집계에만 사용.

이미 반복 분석한 자료의 개발 성적이며 새로운 독립검증이 아니다. F1 .5를 보장하지 않으며 좋은 행만 최종 선정 성능으로 주장하지 않는다. 원래17개 평가와 모집단/구간 기준이 다름. 최종 primary 행은 inner_F1_selected_blend.


In [1]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
s=''.join(json.loads((root/'notebooks/67_rg3_recall_first_nested_mode_ensemble.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s.split('F=pd.DataFrame')[0])
assert len(d)==620 and y.sum()==19 and (y==0).sum()==601
stdcols=[x+'__past10_std' for x in ['Barrel_Temperature_2','Hopper_Temperature','Cushion_Position','Barrel_Temperature_5']]
state=d[['Hopper_Temperature','Barrel_Temperature_2','Barrel_Temperature_5','Cushion_Position','Plasticizing_Time']].copy()
state['pressure_switch_gap']=d.Max_Injection_Pressure-d.Max_Switch_Over_Pressure
memo={};audit=[]
def predict(ref,q):
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 key=tuple(ref)
 if key not in memo:
  normal=ref[y[ref]==0];all_ids=np.arange(len(d))
  im=SimpleImputer(strategy='median',keep_empty_features=True).fit(state.iloc[normal])
  raw=im.transform(state)
  si=SimpleImputer(strategy='median',keep_empty_features=True).fit(d.iloc[ref][stdcols])
  std=si.transform(d[stdcols]);base=np.c_[raw,std]
  normal_ids=normal;norm=base[normal]
  scaler=RobustScaler().fit(norm);context=np.clip(scaler.transform(base),-8,8)
  rank=np.empty_like(base);k=min(75,len(normal)-1)
  for col in range(base.shape[1]):
   other=np.arange(base.shape[1])!=col
   nn=NearestNeighbors(n_neighbors=k+1).fit(context[normal][:,other])
   indices=nn.kneighbors(context[:,other],return_distance=False)
   for row in all_ids:
    ids=indices[row][normal_ids[indices[row]]!=row][:k]
    peers=norm[ids,col]
    rank[row,col]=(1+(peers<base[row,col]).sum()+.5*(peers==base[row,col]).sum())/(len(peers)+2)
  features=[std,rank,base];scores=[];original=None
  for view,F in enumerate(features):
   predictions=[]
   for cls in [RandomForestClassifier,ExtraTreesClassifier]:
    for seed in [42,7]:
     m=cls(n_estimators=200 if view==0 else 80,max_depth=8 if view==0 else 4,min_samples_leaf=3 if view==0 else 4,class_weight='balanced_subsample' if view==0 else 'balanced',random_state=seed,n_jobs=2)
     m.fit(F[ref],y[ref]);p=m.predict_proba(F)[:,1]
     predictions.append(p)
     if view==0 and cls==RandomForestClassifier and seed==42:original=p
   scores.append(np.mean(predictions,axis=0))
  memo[key]=np.column_stack(scores+[original])
  audit.append(dict(train=len(ref),gas_training=int(y[ref].sum()),normal_training=len(normal),train_blocks=','.join(map(str,groups.iloc[ref].unique())),excluded_blocks=','.join(map(str,groups[~groups.index.isin(ref)].unique()))))
 return memo[key][q]
def stats(yy,p):
 tn,fp,fn,tp=confusion_matrix(yy,p,labels=[0,1]).ravel()
 return dict(TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0,recall=tp/(tp+fn) if tp+fn else 0)
def choose_cut(yy,score):
 precision,recall,cuts=precision_recall_curve(yy,score)
 f1=np.divide(2*precision[:-1]*recall[:-1],precision[:-1]+recall[:-1],out=np.zeros_like(cuts),where=(precision[:-1]+recall[:-1])>0)
 order=np.lexsort((cuts,recall[:-1],f1));cut=float(cuts[order[-1]])
 return cut,stats(yy,score>=cut)
# Predeclared simplex weights; pure experts already contain four models.
weights=[np.array([a,b,4-a-b],float)/4 for a in range(5) for b in range(5-a)]
records=[];choices=[];search=[]
for block in blocks:
 q=np.flatnonzero(groups.eq(block));ref=np.flatnonzero(~groups.eq(block))
 out=predict(ref,q);inner=np.full((len(ref),4),np.nan)
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy();inner[mask]=predict(ref[~mask],ref[mask])
 assert np.isfinite(inner).all()
 def record(name,pred,score,cut):
  for j,i in enumerate(q):records.append(dict(model=name,TimeStamp=d.TimeStamp.iloc[i],block=int(block),actual=int(y[i]),prediction=int(pred[j]),score=float(score[j]),threshold=float(cut)))
 candidates=[]
 for w in weights:
  score=inner[:,:3]@w;cut,result=choose_cut(y[ref],score);o=out[:,:3]@w
  candidates.append((result['F1'],result['recall'],-result['FP'],w,cut,o))
  search.append(dict(block=int(block),std_weight=w[0],rank_weight=w[1],state_weight=w[2],threshold=cut,**result))
 chosen=max(candidates,key=lambda z:z[:3])
 record('inner_F1_selected_blend',chosen[5]>=chosen[4],chosen[5],chosen[4])
 choices.append(dict(block=int(block),std_weight=chosen[3][0],rank_weight=chosen[3][1],state_weight=chosen[3][2],threshold=chosen[4],inner_F1=chosen[0],inner_recall=chosen[1]))
 for col,name in enumerate(['deep_std4_ensemble','rank75_ensemble','absolute_state10_ensemble','original_RF_configuration']):
  cut,result=choose_cut(y[ref],inner[:,col]);o=out[:,col]
  record(name+'_inner_F1_cut',o>=cut,o,cut)
  cut=float(np.quantile(inner[y[ref]==0,col],.99,method='higher'))
  record(name+'_normal99_cut',o>cut,o,cut)
 print('Finished RG3 F1-selection block',block,flush=True)
p=pd.DataFrame(records);rows=[]
for name,g in p.groupby('model'):
 row=stats(g.actual,g.prediction);rows.append(dict(model=name,AP=average_precision_score(g.actual,g.score),**row))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(search),'inner_search'),(pd.DataFrame(audit),'training_audit')]:obj.to_csv(tables/f'rg3_F1_joint_selection_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))
print('Frozen 620-row RG3-only development cohort; nested block evaluation, not fresh independent validation.')


Finished RG3 F1-selection block 1


Finished RG3 F1-selection block 2


Finished RG3 F1-selection block 3


Finished RG3 F1-selection block 4


Finished RG3 F1-selection block 5


Finished RG3 F1-selection block 6


Finished RG3 F1-selection block 7


                                 model       AP  TP  FP  FN  TN       F1   recall
absolute_state10_ensemble_inner_F1_cut 0.072270   5 210  14 391 0.042735 0.263158
absolute_state10_ensemble_normal99_cut 0.072270   1  14  18 587 0.058824 0.052632
       deep_std4_ensemble_inner_F1_cut 0.079209   4  50  15 551 0.109589 0.210526
       deep_std4_ensemble_normal99_cut 0.079209   1  11  18 590 0.064516 0.052632
               inner_F1_selected_blend 0.076634   6  48  13 553 0.164384 0.315789
original_RF_configuration_inner_F1_cut 0.080563   6  53  13 548 0.153846 0.315789
original_RF_configuration_normal99_cut 0.080563   1  13  18 588 0.060606 0.052632
          rank75_ensemble_inner_F1_cut 0.076694   5  35  14 566 0.169492 0.263158
          rank75_ensemble_normal99_cut 0.076694   2  10  17 591 0.129032 0.105263


Frozen 620-row RG3-only development cohort; nested block evaluation, not fresh independent validation.
